In [2]:
# PART 1 — Document Ingestion for RAG

# -------- Task 1: Load Documents --------
# Step 1: Import libraries

from langchain_community.document_loaders import PyPDFLoader
import os

# Step 2: Load PDF document
loader = PyPDFLoader("data/ml-book.pdf")

documents = loader.load()

# Step 3: Print details
print("Number of documents/pages:", len(documents))
print("\nSample Content:\n")
print(documents[0].page_content[:500])

Number of documents/pages: 417

Sample Content:

MATHEMATICS  FOR 
MACHINE LEAR NING
Marc Peter Deisenroth
A. Aldo Faisal
Cheng Soon On g
MATHEMATICS FOR MACHINE LEARNING DEISENROTH ET AL.
The fundamental mathematical tools needed to understand machine 
learning include linear algebra, analytic geometry, matrix decompositions, 
vector calculus, optimization, probability and statistics. These topics 
are traditionally taught in disparate courses, making it hard for data 
science or computer science students, or professionals, to efﬁ  ciently le


In [3]:
#  Task 2: Text Splitting 
# Step 1: Import splitter

from langchain_text_splitters import RecursiveCharacterTextSplitter

# Step 2: Configure splitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

# Step 3: Split documents into chunks
chunks = text_splitter.split_documents(documents)

print("Total chunks created:", len(chunks))

Total chunks created: 1215


In [ ]:
# PART 2 — Vector Store & Retriever

#  Task 3: Create Embeddings 
# Step 1: Import Ollama embeddings

from langchain_ollama import OllamaEmbeddings

# Step 2: Initialize embeddings (LOCAL MODEL)
embeddings = OllamaEmbeddings(
    model="nomic-embed-text"
)

#  Task 4: Store Embeddings 
# Step 1: Import Chroma vector store

from langchain_community.vectorstores import Chroma

# Step 2: Create vector database
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory="./chroma_db"
)

# Step 3: Create retriever
retriever = vectorstore.as_retriever(search_kwargs={"k": 4})

print("✅ Vector store created successfully!")

✅ Vector store created successfully!


In [6]:
# PART 3 — Prompt with Message History

#  Task 5: RAG Prompt Template 
# Step 1: Import prompt tools

# Step 1: Import prompt tools (NEW LANGCHAIN IMPORT)

from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
# Step 2: Create prompt template

prompt = ChatPromptTemplate.from_messages([
    ("system",
     """You are a document Q&A assistant.
     Answer ONLY using retrieved context.
     If answer not found, say: I don't know.
     
     Context:
     {context}
     """),

    # Step 3: Message history placeholder
    MessagesPlaceholder(variable_name="chat_history"),

    ("human", "{question}")
])

In [12]:
# ===============================
# PART 4 — Q&A RAG Chain
# ===============================

# -------- Task 6: Build RAG Chain --------

# Step 1: Import required modules (NEW IMPORTS)

from langchain_ollama import ChatOllama
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# Step 2: Initialize LLM (LOCAL OLLAMA MODEL)

llm = ChatOllama(
    model="llama3.2:1b",
    temperature=0
)

# Step 3: Format retrieved documents
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# Step 4: Create Conversational RAG pipeline

from operator import itemgetter

rag_chain = (
    {
        # send ONLY question text to retriever
        "context": itemgetter("question") | retriever | format_docs,

        # pass question separately to prompt
        "question": itemgetter("question"),

        # pass chat history correctly
        "chat_history": itemgetter("chat_history"),
    }
    | prompt
    | llm
    | StrOutputParser()
)

print("✅ RAG Chain Created Successfully")

✅ RAG Chain Created Successfully


In [9]:
#  Task 7: Maintain Message History 

chat_history = []

def ask_question(question):
    global chat_history

    response = rag_chain.invoke({
        "question": question,
        "chat_history": chat_history
    })

    # Store messages
    chat_history.append(("human", question))
    chat_history.append(("ai", response))

    return response

In [10]:
#  Task 8: Trim Chat History 
# Limit by number of messages

MAX_HISTORY = 6

def trim_history():
    global chat_history
    if len(chat_history) > MAX_HISTORY:
        chat_history = chat_history[-MAX_HISTORY:]

In [ ]:
#  Task 9: Multi-turn Testing 

q1 = "What is machine learning?"
print("AI:", ask_question(q1))

trim_history()

q2 = "Explain more about that."
print("AI:", ask_question(q2))

trim_history()

q3 = "What was the previous concept used for?"
print("AI:", ask_question(q3))

AI: I don't know.
AI: Machine learning is a subfield of artificial intelligence (AI) that involves training algorithms to learn from data, without being explicitly programmed or given explicit instructions.

In traditional programming, an algorithm would be written with specific steps and conditions to solve a problem. However, in machine learning, the algorithm learns from data through a process called supervised learning, where it is trained on labeled data, meaning each piece of data is accompanied by a correct output or label.

The algorithm then uses this training data to make predictions or decisions on new, unseen data. This process allows machines to learn patterns and relationships within the data, without being explicitly programmed for every possible scenario.

Machine learning can be applied in various areas, such as:

1. Image recognition: Machines can be trained to recognize objects, faces, or scenes in images.
2. Natural language processing: Machines can be trained to un

In [15]:
#  Task 10: Final Conversational Chatbot 

print("\nConversational RAG Bot (type 'exit' to stop)\n")

while True:
    user_q = input("You: ")

    if user_q.lower() == "exit":
        break

    answer = ask_question(user_q)
    trim_history()

    print("Bot:", answer)


Conversational RAG Bot (type 'exit' to stop)

Bot: Machine learning is the ability of computers to learn from data without being explicitly programmed. It involves training algorithms on large datasets to enable them to make predictions, classify objects, or take actions based on new, unseen data.

Machine learning can be applied in various fields such as:

* Image recognition and classification
* Natural language processing (NLP)
* Predictive modeling
* Recommendation systems
* Robotics

The process of machine learning typically involves the following steps:

1. Data collection: Gathering a large dataset relevant to the problem.
2. Data preprocessing: Cleaning, transforming, and preparing the data for training.
3. Model development: Creating a machine learning model using algorithms such as linear regression or neural networks.
4. Training: Training the model on the prepared data to learn patterns and relationships.
5. Testing: Evaluating the trained model's performance on new, unsee

### Observations & Insights

1. Normal RAG retrieves context once, conversational RAG uses memory.
2. Message history helps resolve follow-up queries.
3. Long memory increases token cost and latency.
4. Trimming improves performance but may reduce context continuity.